In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- expose_date_str ---
FIX_EXPOSE_START_DATE = "2020-01-01"
FIX_EXPOSE_END_DATE = "2020-12-31"

# --- expose_rename ---
FIX_EXPOSE_RENAME_CAL_EXPO = True
FIX_EXPOSE_RENAME_COL_EXPOSURE = "exposure_old"
FIX_EXPOSE_RENAME_COL_POL_NUM = "policy_id"
FIX_EXPOSE_RENAME_COL_POL_PER = "period_old"
FIX_EXPOSE_RENAME_COL_STATUS = "state"
FIX_EXPOSE_RENAME_COLS_DATES = ("issue_dt", "term_dt")
FIX_EXPOSE_RENAME_EXP_COL_POL_PER = "pol_yr"
FIX_EXPOSE_RENAME_EXP_COLS_DATES = ("pol_date_yr", "pol_date_yr_end")
FIX_EXPOSE_RENAME_TRX_RENAMER = lambda x: {"trx_n":"transactions", "trx_amt":"amount"}.get(x, x)
FIX_EXPOSE_RENAME_TRX_TYPES = ["A", "B"]
FIX_EXPOSE_RENAME_PD = pd.DataFrame({
    "policy_id": [1, 2],
    "state": ["Active", "Surrender"],
    "exposure_old": [1.0, 0.5],
    "period_old": [1, 2],
    "issue_dt": ["2020-01-01", "2020-02-01"],
    "term_dt": ["2020-12-31", "2020-11-30"],
    "trx_n": [1, 0],
    "trx_amt": [100.0, 0.0],
})
FIX_EXPOSE_RENAME_PL = pl.from_pandas(FIX_EXPOSE_RENAME_PD)

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_expose_date_str(end_date, start_date):
    end_date = pd.to_datetime(end_date)
    start_date = pd.to_datetime(start_date)
    return start_date, end_date

def before_expose_rename(cal_expo, col_exposure, col_pol_num, col_pol_per, col_status, cols_dates, exp_col_pol_per, exp_cols_dates, trx_renamer, trx_types, data=None):
    if data is None:
        data = pd.DataFrame({"pol_num":[1],"status":["A"],"premium":[100.0]})
    data = data.rename(columns={
        col_pol_num: 'pol_num',
        col_status: 'status',
        col_exposure: 'exposure'
    })

    if not cal_expo and col_pol_per is not None:
        data = data.rename(columns={col_pol_per: exp_col_pol_per})

    if cols_dates is not None:
        data = data.rename(columns={
            cols_dates[0]: exp_cols_dates[0],
            cols_dates[1]: exp_cols_dates[1]
        })

    if trx_types is not None:
        data.columns = [trx_renamer(x) for x in data.columns]
    return data


In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_expose_date_str(end_date, start_date):

    end_date = pl.Series([end_date]).str.to_datetime().item()
    start_date = pl.Series([start_date]).str.to_datetime().item()
    return start_date, end_date

def gen_expose_rename(cal_expo, col_exposure, col_pol_num, col_pol_per, col_status, cols_dates, exp_col_pol_per, exp_cols_dates, trx_renamer, trx_types, data=None):
    if data is None:
        data = pl.DataFrame({"pol_num":[1],"status":["A"],"premium":[100.0]})

    data = data.rename({
        col_pol_num: "pol_num",
        col_status: "status",
        col_exposure: "exposure",
    })

    if not cal_expo and col_pol_per is not None:
        data = data.rename({col_pol_per: exp_col_pol_per})

    if cols_dates is not None:
        data = data.rename({
            cols_dates[0]: exp_cols_dates[0],
            cols_dates[1]: exp_cols_dates[1],
        })

    if trx_types is not None:
        data = data.rename({x: trx_renamer(x) for x in data.columns})
    return data

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: expose_date_str ===

def _same_timestamp_pair(left, right):
    return all(pd.Timestamp(a) == pd.Timestamp(b) for a, b in zip(left, right))

# L1 smoke – generated
try:
    _r = gen_expose_date_str(FIX_EXPOSE_END_DATE, FIX_EXPOSE_START_DATE)
    assert _same_timestamp_pair(_r, (pd.Timestamp("2020-01-01"), pd.Timestamp("2020-12-31")))
    print("✅ L1 smoke gen_expose_date_str: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_expose_date_str: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_expose_date_str(FIX_EXPOSE_END_DATE, FIX_EXPOSE_START_DATE)
    print("✅ L1 smoke before_expose_date_str: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_expose_date_str: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_expose_date_str(FIX_EXPOSE_END_DATE, FIX_EXPOSE_START_DATE)
    _rg = gen_expose_date_str(FIX_EXPOSE_END_DATE, FIX_EXPOSE_START_DATE)
    assert _same_timestamp_pair(_rb, _rg)
    print("✅ L2 equivalence expose_date_str: MATCH")
except Exception as _e:
    print(f"❌ L2 equivalence expose_date_str: setup error — {type(_e).__name__}: {_e}")

# L3 — pandas accepts date objects; generated .str.to_datetime() may not.
try:
    _rb = before_expose_date_str(pd.Timestamp("2020-12-31").date(), pd.Timestamp("2020-01-01").date())
    _rg = gen_expose_date_str(pd.Timestamp("2020-12-31").date(), pd.Timestamp("2020-01-01").date())
    assert _same_timestamp_pair(_rb, _rg)
    print("✅ L3 edge expose_date_str date objects: MATCH")
except Exception as _e:
    print(f"❌ L3 edge expose_date_str date objects: {type(_e).__name__}: {_e}")

# L3 — invalid date strings should fail on both sides.
try:
    before_err = gen_err = None
    try:
        before_expose_date_str("bad-date", "2020-01-01")
    except Exception as e:
        before_err = type(e)
    try:
        gen_expose_date_str("bad-date", "2020-01-01")
    except Exception as e:
        gen_err = type(e)
    if before_err is not None and gen_err is not None and gen_err not in (SyntaxError, NameError):
        print(f"✅ L3 edge expose_date_str invalid string: both sides reject (before={before_err.__name__}, gen={gen_err.__name__})")
    else:
        print(f"❌ L3 edge expose_date_str invalid string: MISMATCH — before={before_err}, gen={gen_err}")
except Exception as _e:
    print(f"❌ L3 edge expose_date_str invalid string: {type(_e).__name__}: {_e}")
